# Optimizing Agentic AI Collaboration v5.1.3 — BoN-MAV
## Verifier: Qwen3-1.7B | Candidates: E2 | Aspects: MMLU-S2 | MMLU

**Model:** `Qwen/Qwen3-1.7B` (verifier only) with 4-bit NF4 quantization via bitsandbytes.
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

## What this notebook is

This notebook runs one **BoN-MAV** configuration: `qwen3-1.7b` verifies the 3-candidate `E2` ensemble (`qwen3-4b` + `ministral-3-3b` + `phi-4-mini`) on **MMLU**, using the **`MMLU-S2`** aspect subset (3 AVs: 1 `direct` + 2 `step_by_step`). This is the **weak-to-strong** — the smallest model judges three larger ones.

Candidates are **replayed** from `E2`'s own SC artifacts (never regenerated) — the exact same `sample_slots(E2, ..., rep=0)` draw the `v4.1.0` Majority Voting notebook uses for `E2`'s own rep 0, so this run's results are **directly, pairwise comparable** to `MAJ@3-E2`'s rep-0 numbers. `MMLU-S2`'s candidate draw is identical to `v5.1.2` (`MMLU-S1`)'s, isolating aspect-subset choice as the only variable between the two runs. Only the verifier is ever live: **9 verifier calls per problem** (3 candidates × 3 aspect verifiers), each an independent `chain.invoke` (no shared KV-cache across aspects).

## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocol — BoN-MAV.** Consumes `E2`'s SC generation artifacts directly (no regeneration); paired against `v4.1.0`'s `MAJ@3-E2` run and against `v5.1.2`'s `MMLU-S1` run for protocol- and subset-comparisons at identical candidate cost. Latency/token/VRAM (via `utils.voting.compose_generation_metrics`, split `reused_gen_*`/`live_verify_*`) feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add secrets via **Add-ons → Secrets**: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_API_KEY`.
3. **Before clicking Run All**, fill in §3's `PRODUCER_REFS` (pinned SC artifact versions for `qwen3-4b`, `ministral-3-3b`, `phi-4-mini` on `mmlu`) and, if resuming a killed session, `RESUME_FROM_REF`.
4. Click **Run All**. If the session is killed, note the last-printed re-log ref, set `RESUME_FROM_REF` to it, and re-run — the loop skips every already-logged `problem_id`.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets**: `HF_TOKEN`, `WANDB_API_KEY`, `GITHUB_TOKEN` (Notebook access enabled)
3. Fill in `PRODUCER_REFS`/`RESUME_FROM_REF` in §3 before running, same as above.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `langgraph`, `datasets`, `wandb`, `math-verify`). If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [1]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

Cloning into '/kaggle/working/tfg-agentic-ai'...


✅ Repo cloned to /kaggle/working/tfg-agentic-ai
repo_root = /kaggle/working/tfg-agentic-ai


In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Config

`ENSEMBLE` (`E2`) and `SUBSET_ID` (`MMLU-S2`) fix this notebook's configuration; `PRODUCER_REFS` pins the 3 candidate models' SC artifacts **explicitly** (`entity/project/name:vN`, never `:latest`). `RESUME_FROM_REF` — leave `None` for a fresh run; set to this notebook's own last-printed verdicts-artifact ref to resume a killed session.

In [4]:
import random

from utils.ensembles import E2, SLOT_SAMPLE_SEED, bonmav_matrices_artifact_name, bonmav_run_name, sample_slots, slot_agent_ids
from utils.registry import MODELS, generations_artifact_name, get_model, problem_ref
from utils.mav import MAV_ASPECT_SUBSETS, MAV_ASPECTS, av_position_labels

VERIFIER_KEY = "qwen3-1.7b"
ENSEMBLE = E2
SUBSET_ID = "S2"
DATASET = "mmlu"
ASPECT_IDS = MAV_ASPECT_SUBSETS[DATASET][SUBSET_ID]
AV_LABELS = av_position_labels(ASPECT_IDS)
AV_INTENTIONS = {label: MAV_ASPECTS[DATASET][aid]["intention"] for aid, label in AV_LABELS.items()}

SAMPLE_PREFIX = "math_sample"

PROJECT = "agentic-final-results" if (ENVIRONMENT in ("kaggle", "colab")) else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

WANDB_ENTITY = "tfg-optimizing-agentic-ai"
SC_PROJECT = "agentic-final-results"

CANDIDATE_MODEL_KEYS = list(dict.fromkeys(ENSEMBLE.slots))
PRODUCER_REFS = {}
for model_key in CANDIDATE_MODEL_KEYS:
    version = "v9" if "ministral" in model_key else "v4"
    PRODUCER_REFS[model_key] = f"{WANDB_ENTITY}/{SC_PROJECT}/{generations_artifact_name(model_key, DATASET)}:{version}"

DATA_SEED = 789
EVAL_SIZE = 128
GEN_COMMON = {"temperature": 0.7, "top_p": 0.95, "top_k": 40}  # matches the L1 GEN params
TIE_BREAK_SEED = 2026
LOG_EVERY_N_PROBLEMS = 10  # re-log the resumable verdicts artifact this often

# Set to this notebook's own last-printed verdicts-artifact ref to resume a killed
# session instead of restarting at problem_id=0; leave None for a fresh run.
RESUME_FROM_REF = None

VERDICTS_ARTIFACT_NAME = f"l1-bonmav-{VERIFIER_KEY}-{ENSEMBLE.ensemble_id}-{SUBSET_ID}-{DATASET}-verifications"

print(f"Verifier: {VERIFIER_KEY}  |  Candidates: {ENSEMBLE.ensemble_id} {list(ENSEMBLE.slots)}  |  "
      f"Subset: {SUBSET_ID}  |  Dataset: {DATASET}")
print(f"Aspect ids ({len(ASPECT_IDS)}): {ASPECT_IDS}")
print(f"AV labels: {AV_LABELS}  |  AV intentions: {AV_INTENTIONS}")
print(f"\nPinned candidate producer refs:")
for mk, ref in PRODUCER_REFS.items():
    print(f"  {mk}: {ref}")

W&B Project: agentic-final-results
Verifier: qwen3-1.7b  |  Candidates: E2 ['qwen3-4b', 'ministral-3-3b', 'phi-4-mini']  |  Subset: S2  |  Dataset: mmlu
Aspect ids (3): ['mmlu-s2-direct', 'mmlu-s2-logical-soundness', 'mmlu-s2-factual-correctness']
AV labels: {'mmlu-s2-direct': 'AV1', 'mmlu-s2-logical-soundness': 'AV2', 'mmlu-s2-factual-correctness': 'AV3'}  |  AV intentions: {'AV1': 'direct', 'AV2': 'logical_soundness', 'AV3': 'factual_correctness'}

Pinned candidate producer refs:
  qwen3-4b: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-qwen3-4b-mmlu:v4
  ministral-3-3b: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-ministral-3-3b-mmlu:v9
  phi-4-mini: tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-phi-4-mini-mmlu:v4


## 4. Data Loading

The evaluation subset is drawn with `random.seed(789)`, identical to every SC/MAJ notebook's own subset — so `problem_id` identity and `sample_slots`' draws all agree across protocols.

In [5]:
from utils.data import load_mmlu
from utils.voting import MMLU_SPEC

SPEC = MMLU_SPEC
mmlu_dataset = load_mmlu(split="test")

random.seed(DATA_SEED)
test_sample = random.sample(list(mmlu_dataset), EVAL_SIZE)

print(f"✅ MMLU dataset loaded with {len(mmlu_dataset)} examples.")
print(f"✅ {len(test_sample)} questions sampled (seed={DATA_SEED})")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

✅ MMLU dataset loaded with 14042 examples.
✅ 128 questions sampled (seed=789)


## 5. Reuse — Candidate Model Keys

`CANDIDATE_MODEL_KEYS` (from §3) is every distinct model in `ENSEMBLE`. The actual producer-artifact fetch (`utils.persistence.fetch_and_validate_stores`: `load_generations_artifact` → `build_generation_store` → `validate_alignment`, the same reuse pattern `v4.1.0` and the `v3.0.0` smoke test's round-trip stage already use) now happens in §8, immediately after the real BoN-MAV run is opened, via `run=run` — so the run that actually consumes the data also records its own producer→consumer lineage, instead of an earlier anonymous download. No model is loaded here — this is pure download + validation, deferred to §8.

## 6. Load Verifier Model

Only the verifier is ever resident in VRAM — every candidate is replayed, never regenerated — so `batch_size=1` and the single-model VRAM budget hold throughout, identical in shape to the SC notebooks' own model-loading cell.

In [6]:
from utils.models import load_model_and_tokenizer
from utils.warmup import warmup_model

verifier_cfg = get_model(VERIFIER_KEY)
model, tokenizer = load_model_and_tokenizer(verifier_cfg["model_id"], quantize_4bit=True)
print(f"✅ Verifier {VERIFIER_KEY} loaded on {model.device} — dtype {model.dtype}")

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"],
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'top_k', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Verifier qwen3-1.7b loaded on cuda:0 — dtype torch.float16
🔥 Warming up model (2 iterations, max_new_tokens=256)…


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Warmup complete in 34.3s — first evaluated sample will no longer pay cold-start cost.


34.258560189000036

## 7. Verify Function and Select Function

`verify_fn` is a `timed_invoke`-based closure over the loaded verifier: it renders the transcribed aspect prompt (`utils.mav.MAV_ASPECTS`) with `{task}`/`{candidate}`, dispatching on the aspect's `strategy` for `max_new_tokens` (128 `direct` / 512 `reasoning` / 1024 `step_by_step`). This subset's 3 AVs use only the `direct` and `step_by_step` tiers — `reasoning` is not exercised here. `select_fn` wraps `utils.mav.select_bon_candidate` with this benchmark's `SPEC` bound (`build_bon_mav_graph` calls it as `select_fn(n_candidates, approvals, candidate_answers, rng=rng)`, with no `spec` parameter of its own).

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from utils.agents import build_hf_langchain_llm
from utils.instrumentation import timed_invoke, tokens_per_second
from utils.mav import MAV_ASPECTS, parse_verdict, select_bon_candidate


def make_verify_fn(model, tokenizer, model_key, dataset_key, sampling):
    """A timed_invoke-based closure over the (single, already-loaded) verifier
    model, dispatching on the aspect's strategy for max_new_tokens — one llm
    built per distinct max_new_tokens value, reused across problems/aspects
    that share it."""
    llm_by_max_tokens = {}

    def get_llm(max_new_tokens):
        if max_new_tokens not in llm_by_max_tokens:
            llm_by_max_tokens[max_new_tokens] = build_hf_langchain_llm(
                model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens,
                temperature=sampling["temperature"], top_p=sampling["top_p"], top_k=sampling["top_k"],
                return_full_text=False,
            )
        return llm_by_max_tokens[max_new_tokens]

    def verify(task, candidate_answer, aspect_id):
        aspect = MAV_ASPECTS[dataset_key][aspect_id]
        prompt_template = PromptTemplate.from_template(aspect["prompt"])
        llm = get_llm(aspect["max_new_tokens"])
        chain = prompt_template | llm | StrOutputParser()
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task, "candidate": candidate_answer})
        return {
            "candidate_idx": -1, "aspect_id": aspect_id, "verdict": parse_verdict(r["output_text"]),
            "raw_output": r["output_text"], "latency_s": r["latency_s"],
            "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
            "total_tokens": r["total_tokens"], "gpu_peak_mb": r["gpu_peak_mb"],
            "model_key": model_key,
        }
    return verify


def make_select_fn(spec):
    def select_fn(n_candidates, approvals, candidate_answers, *, rng):
        return select_bon_candidate(n_candidates, approvals, candidate_answers, spec=spec, rng=rng)
    return select_fn


verify_fn = make_verify_fn(model, tokenizer, VERIFIER_KEY, DATASET, GEN_COMMON)
select_fn = make_select_fn(SPEC)
print("✅ verify_fn and select_fn ready.")

✅ verify_fn and select_fn ready.


## 8. Resumable BoN-MAV Run Loop

`run = init_wandb(...)` opens the real run first, then `fetch_and_validate_stores(CANDIDATE_MODEL_KEYS, ..., run=run)` downloads + validates every candidate producer artifact — real producer→consumer lineage recorded on *this* run (`run.config` is stamped with `alignment_validated`/`alignment_record_counts` afterward via `run.config.update(...)`, since those aren't known before the fetch). `RESUME_FROM_REF`, if set, is then loaded the same way (`load_generations_artifact(RESUME_FROM_REF, run=run)`). Per problem: `sample_slots(ENSEMBLE, ..., rep=0)` (the same rep-0 draw `v4.1.0`'s `MAJ@3-E2` uses) → 3 candidate `AgentSpec`s (`mode="replay"`) → `build_bon_mav_graph(...)` → `graph.invoke(...)` → score `out["selection"]`'s pick against gold **outside** the graph → `compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")`, plus the `reused_gen_*`/`live_verify_*` split computed separately on each half.

**Resumable, not just checkpointed**: every `LOG_EVERY_N_PROBLEMS` problems (and once more at the end), the full record list so far is re-logged under one growing artifact name (`log_generations_artifact` — the identical crash-resumption mechanism the SC generation pass uses per-seed). If this session is killed, the last-printed ref is the one to set as `RESUME_FROM_REF` (§3) — the loop then skips every already-logged `problem_id`.

In [8]:
import time
import wandb
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics, log_metrics, log_stat_summary, finish_wandb
from utils.graphs import AgentSpec, build_bon_mav_graph
from utils.voting import compose_generation_metrics
from utils.persistence import artifact_ref, load_generations_artifact, log_generations_artifact, fetch_and_validate_stores
from utils.mav import candidate_correctness, per_problem_av_metrics

# Protocol tag format: {protocol name}@N/M<N> = BoN-MAV with N=3 generators
# (candidates) and M=3 Aspect Verifiers per subset (see utils/mav.py). The
# logged tag string itself is unchanged.
run = init_wandb(
    run_name=bonmav_run_name(VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET),
    project=PROJECT,
    config={
        "protocol": "BoN-MAV@3/M3", "verifier_key": VERIFIER_KEY, "candidate_ensemble": ENSEMBLE.ensemble_id,
        "aspect_subset": SUBSET_ID, "aspect_ids": ASPECT_IDS, "av_intentions": AV_INTENTIONS,
        "eval_dataset": DATASET, "eval_size": EVAL_SIZE,
        "data_seed": DATA_SEED, "tie_break_seed": TIE_BREAK_SEED, "slot_sample_seed": SLOT_SAMPLE_SEED,
        "producer_artifacts": PRODUCER_REFS, "resumed_from": RESUME_FROM_REF,
    },
    tags=["l1", "bon-mav", VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET],
)
# Own W&B run fetches its own copy of every candidate producer artifact via
# load_generations_artifact(run=run) — real producer->consumer lineage
# recorded on THIS run, not a bare use_artifact call. alignment_validated/
# _record_counts aren't known until after the fetch, so they're stamped via
# run.config.update(...) rather than the initial config= dict above.
stores, alignment_counts = fetch_and_validate_stores(
    CANDIDATE_MODEL_KEYS, lambda mk: PRODUCER_REFS[mk],
    run=run, dataset_sample=test_sample, render_task=SPEC.render_task,
    expected_dataset=DATASET, expected_data_seed=DATA_SEED,
    expected_eval_size=EVAL_SIZE, n_generators=5,
)
run.config.update({
    "alignment_validated": True,
    "alignment_record_counts": alignment_counts,
})
define_wandb_chart_only_metrics(prefixes=[f"{SAMPLE_PREFIX}/*"])

bonmav_records = []
already_done = set()

if RESUME_FROM_REF:
    resumed_records, _ = load_generations_artifact(RESUME_FROM_REF, run=run)
    bonmav_records = resumed_records
    already_done = {r["problem_id"] for r in resumed_records}
    print(f"Resuming from {RESUME_FROM_REF}: {len(already_done)} problems already logged, skipping them.")
else:
    print("Starting fresh at problem_id=0.")

bonmav_tie_rng = random.Random(TIE_BREAK_SEED)
agent_ids = slot_agent_ids(ENSEMBLE)
n_aspects = len(ASPECT_IDS)
producer_ref = None


def _log_manifest():
    return {
        "verifier_key": VERIFIER_KEY, "dataset": DATASET,
        "candidate_ensemble": ENSEMBLE.ensemble_id, "aspect_subset": SUBSET_ID,
        "protocol": "BoN-MAV@3/M3", "data_seed": DATA_SEED, "eval_size": EVAL_SIZE,
        "wandb_run_id": run.id, "wandb_run_name": run.name, "wandb_project": PROJECT,
        "n_records": len(bonmav_records),
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


for problem_id, item in enumerate(test_sample):
    if problem_id in already_done:
        continue

    slots = sample_slots(ENSEMBLE, dataset=DATASET, problem_id=problem_id, rep=0)
    candidate_specs = [
        AgentSpec(agent_id=aid, model_key=mk, model_id=get_model(mk)["model_id"], mode="replay",
                  store=stores[mk], gen_id=gid)
        for aid, (mk, gid) in zip(agent_ids, slots)
    ]
    graph = build_bon_mav_graph(candidate_specs, ASPECT_IDS, verify_fn, select_fn, rng=bonmav_tie_rng)
    task = SPEC.render_task(item)
    ground_truth = SPEC.get_truth(item)
    out = graph.invoke({
        "task": task, "problem_id": problem_id, "dataset": DATASET,
        "samples": [], "approvals": [], "selection": None,
    })

    selection = out["selection"]
    selected_idx = selection["selected_idx"]
    selected_sample = out["samples"][selected_idx]
    compare_value = selected_sample["answer"] if SPEC.compare_on == "answer" else selected_sample["parsed_output"]
    correct = SPEC.score(compare_value, ground_truth)

    # Scored against every candidate (not just the selected one) so per-AV
    # accuracy — (verdict is True) == candidate_correct[candidate_idx] — can
    # be computed per problem here and reused, unrecomputed, in §9.
    candidate_correct = candidate_correctness(out["samples"], ground_truth, SPEC)
    av_metrics = per_problem_av_metrics(out["approvals"], AV_LABELS, candidate_correct)

    # reused_gen_*/live_verify_* cost split computed separately on each
    # half, plus the combined total the results/matrices use.
    reused_metrics = compose_generation_metrics(out["samples"], tokens_per_s_rule="mean")
    live_metrics = compose_generation_metrics(out["approvals"], tokens_per_s_rule="mean")
    composed = compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")

    record = {
        "dataset": DATASET, "problem_id": problem_id,
        "verifier_key": VERIFIER_KEY, "candidate_ensemble": ENSEMBLE.ensemble_id, "aspect_subset": SUBSET_ID,
        "protocol": "BoN-MAV@3/M3", "slots": [[mk, gid] for mk, gid in slots],
        "selected_idx": selected_idx, "scores": selection["scores"],
        "tie": selection["tie"], "tie_break_used": selection["tie_break_used"],
        "ground_truth": ground_truth, "final_output": selected_sample["parsed_output"], "correct": correct,
        "pct_approvals_top": selection["scores"][selected_idx] / n_aspects,
        "approvals": out["approvals"],
        "candidate_correct": candidate_correct, "av_metrics": av_metrics,
        "latency_s": composed["latency_s"], "output_tokens": composed["output_tokens"],
        "total_tokens": composed["total_tokens"], "gpu_peak_mb": composed["gpu_peak_mb"],
        "throughput_tok_s": composed["tokens_per_s"],
        "reused_gen_latency_s": reused_metrics["latency_s"], "reused_gen_total_tokens": reused_metrics["total_tokens"],
        "live_verify_latency_s": live_metrics["latency_s"], "live_verify_total_tokens": live_metrics["total_tokens"],
    }
    bonmav_records.append(record)

    av_payload = {}
    for label, m in av_metrics.items():
        av_payload[f"{SAMPLE_PREFIX}/{label}_latency_s"] = round(m["latency_s"], 2)
        av_payload[f"{SAMPLE_PREFIX}/{label}_output_tokens"] = round(m["output_tokens"], 2)
        av_payload[f"{SAMPLE_PREFIX}/{label}_accuracy"] = round(m["accuracy"], 4)

    log_metrics({
        f"{SAMPLE_PREFIX}/accuracy": int(correct),
        f"{SAMPLE_PREFIX}/latency_s": round(composed["latency_s"], 2),
        f"{SAMPLE_PREFIX}/output_tokens": composed["output_tokens"],
        f"{SAMPLE_PREFIX}/total_tokens": composed["total_tokens"],
        f"{SAMPLE_PREFIX}/gpu_peak_mb": round(composed["gpu_peak_mb"], 2),
        f"{SAMPLE_PREFIX}/throughput_tok_s": round(composed["tokens_per_s"], 4),
        **av_payload,
    })
    print(f"  problem={problem_id}: selected_idx={selected_idx} correct={correct} "
          f"tie={selection['tie']} scores={selection['scores']}")

    if len(bonmav_records) % LOG_EVERY_N_PROBLEMS == 0:
        artifact = log_generations_artifact(bonmav_records, _log_manifest(), name=VERDICTS_ARTIFACT_NAME, run=run)
        producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
        wandb.summary["producer_artifact"] = producer_ref
        print(f"  ✅ re-logged verdicts artifact after {len(bonmav_records)} problems -> {producer_ref}")

# Final re-log to catch any tail not a multiple of LOG_EVERY_N_PROBLEMS.
artifact = log_generations_artifact(bonmav_records, _log_manifest(), name=VERDICTS_ARTIFACT_NAME, run=run)
producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
wandb.summary["producer_artifact"] = producer_ref

print(f"\n✅ BoN-MAV generation complete: {len(bonmav_records)}/{EVAL_SIZE} problems logged -> {producer_ref}")
if len(bonmav_records) < EVAL_SIZE:
    print(
        f"⚠️  Only {len(bonmav_records)}/{EVAL_SIZE} problems completed this session. "
        f"Set RESUME_FROM_REF = {producer_ref!r} in §3 and re-run to finish the remaining "
        f"{EVAL_SIZE - len(bonmav_records)} problems."
    )

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260828_171204-w8rtkgmh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run l1-bonmav-qwen3-1.7b-E2-S2-mmlu
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-

Starting fresh at problem_id=0.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.c

  problem=0: selected_idx=0 correct=True tie=False scores=[3, 0, 2]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.c

...

  problem=125: selected_idx=0 correct=False tie=False scores=[3, 0, 2]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.c

  problem=126: selected_idx=0 correct=False tie=False scores=[3, 0, 0]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.c

  problem=127: selected_idx=0 correct=True tie=True scores=[3, 2, 3]

✅ BoN-MAV generation complete: 128/128 problems logged -> tfg-optimizing-agentic-ai/agentic-final-results/l1-bonmav-qwen3-1.7b-E2-S2-mmlu-verifications:v19


## 9. Final Statistics, Matrices Artifact, and Analysis

**Only meaningful once all `EVAL_SIZE` problems have been logged** (across however many sessions it took) — if this session stopped early, resume per the warning printed at the end of §8 and re-run this cell afterwards rather than treating a partial result as final.

In [9]:
import gc
import tempfile
import numpy as np
import torch
from utils.statistics import (
    accuracy_statistics, time_statistics, token_statistics, vram_statistics, pct_votes_top_statistics,
    print_accuracy_stats_report, print_time_report, print_token_report, print_vram_report, print_pct_votes_top_report,
)
from utils.mav import MAV_ASPECTS, approval_diagnostics, verifier_quality_diagnostics

if len(bonmav_records) < EVAL_SIZE:
    print(f"⚠️  Only {len(bonmav_records)}/{EVAL_SIZE} problems completed — the analysis below "
          f"is PARTIAL, not the final result. Resume and re-run once all problems are done.")

sorted_records = sorted(bonmav_records, key=lambda r: r["problem_id"])
n_done = len(sorted_records)

correctness_matrix = np.zeros((n_done, 1), dtype=int)
time_matrix         = np.zeros((n_done, 1), dtype=float)
output_tok_matrix   = np.zeros((n_done, 1), dtype=int)
total_tok_matrix    = np.zeros((n_done, 1), dtype=int)
gpu_peak_matrix      = np.zeros((n_done, 1), dtype=float)
pct_approvals_top_matrix = np.zeros((n_done, 1), dtype=float)

# Per-AV (AV1/AV2/AV3) latency/output-token matrices, filled from the
# per-problem averages §8 already computed and persisted onto each record —
# reused here for avg+CI, never recomputed.
av_matrices = {
    label: {"latency_s": np.zeros((n_done, 1), dtype=float), "output_tokens": np.zeros((n_done, 1), dtype=float)}
    for label in AV_LABELS.values()
}

results_columns = [
    "dataset", "verifier", "candidate_ensemble", "aspect_subset", "protocol", "problem_id", "problem_ref",
    "task", "slots", "selected_idx", "ground_truth", "final_output", "correct", "pct_approvals_top", "tie",
    "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb", "throughput_tok_s",
]
if DATASET == "math500":
    results_columns += ["subject", "level"]
results_rows = []

# One row per verifier call (problem x candidate x aspect) — candidate answer
# text isn't persisted anywhere per-call, so it's re-derived here from the
# same replay stores §8 used; candidate_correct itself is already on each
# record (computed per-problem in §8).
verifier_calls_columns = [
    "problem_id", "verifier", "aspect_id", "strategy", "candidate_idx", "candidate_model_key",
    "candidate_answer", "candidate_correct", "verdict", "raw_output", "ground_truth",
    "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb",
]
verifier_calls_rows = []
all_approvals = []

for i, r in enumerate(sorted_records):
    correctness_matrix[i, 0] = int(r["correct"])
    time_matrix[i, 0]        = r["latency_s"]
    output_tok_matrix[i, 0]  = r["output_tokens"]
    total_tok_matrix[i, 0]   = r["total_tokens"]
    gpu_peak_matrix[i, 0]    = r["gpu_peak_mb"]
    pct_approvals_top_matrix[i, 0] = r["pct_approvals_top"]
    for label, m in r["av_metrics"].items():
        av_matrices[label]["latency_s"][i, 0] = m["latency_s"]
        av_matrices[label]["output_tokens"][i, 0] = m["output_tokens"]

    item = test_sample[r["problem_id"]]
    candidate_records = [stores[mk][(r["problem_id"], gid)] for mk, gid in r["slots"]]
    candidate_correct = r["candidate_correct"]  # already computed per-problem in §8

    for a in r["approvals"]:
        candidate_idx = a["candidate_idx"]
        candidate_mk, _ = r["slots"][candidate_idx]
        a["candidate_correct"] = candidate_correct[candidate_idx]
        verifier_calls_rows.append([
            r["problem_id"], VERIFIER_KEY, a["aspect_id"], MAV_ASPECTS[DATASET][a["aspect_id"]]["strategy"],
            candidate_idx, candidate_mk,
            candidate_records[candidate_idx]["parsed_output"], candidate_correct[candidate_idx],
            a["verdict"], a["raw_output"], r["ground_truth"],
            round(a["latency_s"], 2), a["output_tokens"], a["total_tokens"],
            round(a["gpu_peak_mb"], 2),
        ])
    all_approvals.extend(r["approvals"])

    slots_str = ",".join(f"{mk}:{gid}" for mk, gid in r["slots"])
    row = [
        DATASET, VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, r["protocol"], r["problem_id"],
        problem_ref(DATASET, r["problem_id"]), SPEC.render_task(item), slots_str, r["selected_idx"], r["ground_truth"],
        r["final_output"], r["correct"], round(r["pct_approvals_top"], 4), r["tie"],
        round(r["latency_s"], 2), r["output_tokens"], r["total_tokens"],
        round(r["gpu_peak_mb"], 2), round(r["throughput_tok_s"], 4),
    ]
    if DATASET == "math500":
        row += [item["subject"], item["level"]]
    results_rows.append(row)

results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({f"results_bonmav_{VERIFIER_KEY}_{ENSEMBLE.ensemble_id}_{SUBSET_ID}": results_table})

verifier_calls_table = wandb.Table(columns=verifier_calls_columns, data=verifier_calls_rows)
wandb.log({f"results_bonmav_calls_{VERIFIER_KEY}_{ENSEMBLE.ensemble_id}_{SUBSET_ID}": verifier_calls_table})

acc_stats   = accuracy_statistics(correctness_matrix)
time_stats  = time_statistics(time_matrix)
out_stats   = token_statistics(output_tok_matrix)
total_stats = token_statistics(total_tok_matrix)
vram_stats  = vram_statistics(gpu_peak_matrix)
pct_approvals_top_stats = pct_votes_top_statistics(pct_approvals_top_matrix)

print_accuracy_stats_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Accuracy ({DATASET})", acc_stats)
print_time_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Latency ({DATASET})", time_stats)
print_vram_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Peak VRAM ({DATASET})", vram_stats)
print_pct_votes_top_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Winner Approval Share ({DATASET})", pct_approvals_top_stats)

n_tie = sum(1 for r in sorted_records if r["tie"])
diag = approval_diagnostics(all_approvals, n_candidates=len(ENSEMBLE.slots))
quality_diag = verifier_quality_diagnostics(all_approvals)

wandb.summary["accuracy"]            = round(acc_stats["p_hat"], 6)
wandb.summary["selection_tie_rate"]  = round(n_tie / n_done, 4)
wandb.summary["mean_vram_peak"]      = round(vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]       = round(vram_stats["peak_max"], 0)
wandb.summary["n_problems_completed"] = n_done
wandb.summary["mean_pct_approvals_top"] = round(pct_approvals_top_stats["mean_pct_votes_top"], 4)

# parse_rate only — approval_rate stays a console-only sycophancy flag, not a
# BoN-MAV summary statistic. Keyed by AV1/AV2/AV3, not the raw aspect_id, so
# it's comparable across subsets (see av_position_labels).
for aspect_id, stats in diag.items():
    label = AV_LABELS[aspect_id]
    wandb.summary[f"parse_rate/{label}"] = round(stats["parse_rate"], 4)
    print(f"  {label} ({aspect_id}): approval_rate={stats['approval_rate']:.1%} parse_rate={stats['parse_rate']:.1%}")
    if stats["approval_rate"] > 0.95:
        print(f"    ⚠️  sycophancy flag: {label} ({aspect_id}) approves >95% of candidates")

# Whether the verifier is actually right to approve, not just how often it
# does — the counterpart to the sycophancy flag above.
for aspect_id, qstats in quality_diag.items():
    label = AV_LABELS[aspect_id]
    wandb.summary[f"verifier_accuracy/{label}"] = round(qstats["accuracy"], 4)
    if qstats["precision"] is not None:
        wandb.summary[f"verifier_precision/{label}"] = round(qstats["precision"], 4)
    if qstats["recall"] is not None:
        wandb.summary[f"verifier_recall/{label}"] = round(qstats["recall"], 4)
    if qstats["f1"] is not None:
        wandb.summary[f"verifier_f1/{label}"] = round(qstats["f1"], 4)
    print(f"  {label} ({aspect_id}): verifier_accuracy={qstats['accuracy']:.1%} "
          f"precision={qstats['precision']} recall={qstats['recall']} f1={qstats['f1']}")

mean_live_latency = sum(r["live_verify_latency_s"] for r in sorted_records) / n_done
wandb.summary["mean_live_verify_latency_s"]  = round(mean_live_latency, 2)

log_stat_summary("accuracy_stats", acc_stats, keys=["ci_lower", "ci_upper", "between_problem_std"])
log_stat_summary("latency_stats", time_stats, keys=["mean", "ci_lower", "ci_upper"])
log_stat_summary("output_tokens_stats", out_stats, keys=["mean", "ci_lower", "ci_upper"])
log_stat_summary("total_tokens_stats", total_stats, keys=["mean", "ci_lower", "ci_upper"])
log_stat_summary("vram_peak_stats", vram_stats, keys=["peak_ci_lower", "peak_ci_upper"])
log_stat_summary("pct_approvals_top_stats", pct_approvals_top_stats, keys=["ci_lower", "ci_upper", "between_problem_std"])

# Per-AV (AV1/AV2/AV3) latency/output-token avg+CI — pure reuse of the same
# time_statistics/token_statistics/log_stat_summary already used above,
# applied per AV instead of pooled over the whole run.
for label, mats in av_matrices.items():
    av_latency_stats = time_statistics(mats["latency_s"])
    av_output_tokens_stats = token_statistics(mats["output_tokens"])
    log_stat_summary(f"{label}_latency_stats", av_latency_stats, keys=["mean", "ci_lower", "ci_upper"])
    log_stat_summary(f"{label}_output_tokens_stats", av_output_tokens_stats, keys=["mean", "ci_lower", "ci_upper"])

# ── Evaluation-matrices artifact ────────────────────────────────────────────
artifact = wandb.Artifact(
    bonmav_matrices_artifact_name(VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET),
    type="evaluation-data",
    description=f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} evaluation matrices "
                f"({n_done} x 1). {DATASET}.",
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        problem_ids=np.array([r["problem_id"] for r in sorted_records]),
        bonmav_correctness=correctness_matrix,
        bonmav_times=time_matrix,
        bonmav_output_tokens=output_tok_matrix,
        bonmav_total_tokens=total_tok_matrix,
        bonmav_gpu_peak=gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)

# ── Unload verifier ──────────────────────────────────────────────────────────
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
print(f"\nVRAM after unloading verifier: {final_allocated_mb:.1f} MB")

finish_wandb()
print(f"\n✅ BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID}/{DATASET} run finished: "
      f"accuracy={acc_stats['p_hat']:.3f} ({n_done}/{EVAL_SIZE} problems)")


  BoN-MAV qwen3-1.7b/E2/S2 Accuracy (mmlu)
  Accuracy p̂                  59.38%
  95% Cluster-t CI         [50.75, 68.00]%
  Between-problem std        49.3062%
  ── Supplementary: Wilson ──
  95% Wilson CI              [50.68, 67.52]%
  DEFF                       127.000
  ICC                          0.000
  ── Sample ──
  Problems x Seeds           128 x 1  (n=128)


  BoN-MAV qwen3-1.7b/E2/S2 Latency (mmlu)
  Mean (s)                    673.7991
  Std (s)                     168.1204
  CV                            0.2495
  95% BCa CI (s)             [644.7809, 702.0139]  [cluster_bca]
  Between-problem std (s)     168.1204
  ── Percentiles (descriptive) ──
  Median / P50 (s)            664.6247
  P25 (s)                     554.4361
  P75 (s)                     810.6324
  IQR (s)                     256.1963
  P95 (s)                     925.7259
  ── Sample ──
  Problems x Seeds           128 x 1  (n=128)


  BoN-MAV qwen3-1.7b/E2/S2 Peak VRAM (mmlu) — Peak VRAM (MB)
  Mean pe

wandb: uploading artifact run-w8rtkgmh-results_bonmav_qwen3-1.7b_E2_S2; uploading artifact run-w8rtkgmh-results_bonmav_calls_qwen3-1.7b_E2_S2; uploading artifact evaluation-matrices-bonmav-qwen3-1.7b-E2-S2-mmlu; updating run metadata



VRAM after unloading verifier: 1385.1 MB


wandb: uploading media/table/results_bonmav_qwen3-1.7b_E2_S2_128_f211cf2a2fb17365007e.table.json; uploading media/table/results_bonmav_calls_qwen3-1.7b_E2_S2_129_c8897518bf98b1d7bc2b.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:      math_sample/AV1_accuracy ▃▆█▆▃▆▃▆▃▃▃▁▆▃▃▃▆▃▆▆▃▆▆▃▆▆▆▃▆▆▃▁▆▆▃▆█▃█▃
wandb:     math_sample/AV1_latency_s ▁▄▄▁▄▄█▁█▁█▁█▄▄▄█▄▄▄▁▁▁█▄▄▄▄▁▁▁▄███▁▁█▁▅
wandb: math_sample/AV1_output_tokens ▁▅▅▁█▅▅▁▁▅▁▁█▅▁▁▁█▅▁▁▅▁▁▅▅▅▁▅▅▁█▁▅▁▅▁▁▁▅
wandb:      math_sample/AV2_accuracy ▆▃▁█▆█▆▆█▆▆▁▆▁███▃██▃▆▁▆█▆█▃█▆██▃▆▃███▁█
wandb:     math_sample/AV2_latency_s ▅▂▅▃▆▃▇▇▄▄▄▅▃▅▃▃▄▄▆▆▅▅▆▅▄▁▃▄▃▄▃▅▆▄█▅▆▅▄▄
wandb: math_sample/AV2_output_tokens ▃▂▅▆▆▃▂▄█▅▆▄▆▃▃▂▆▆▅▅▅▅▄▃▇▂▆▄▄▃▅▄▆▆▄▅▁▄▆▅
wandb:      math_sample/AV3_accuracy ▃▁██▆██▆▆▃▆▁▆▆▃▃█▆██▆▃▃▆▁▆█▃█▆▆█▃█▃▁▆█▆█
wandb:     math_sample/AV3_latency_s ▄▇▆▇▄▄▆█▂█▇▃▄▅▃▁▃▅▁▃█▆▅▅▃▅▆▅▇▆▃▅▇▆▄▃▆▂▆▄
wandb: math_sample/AV3_output_tokens ▆▇▇▇▂▅█▂▇▆▅▆▆▅▅▃▅▆▄▇▇▃▃▅▅▄▂▇▄▆▆▇▅▃▄▆▅▁▃▂
w


✅ BoN-MAV qwen3-1.7b/E2/S2/mmlu run finished: accuracy=0.594 (128/128 problems)
